# Laya-JA：エンコーダーを動かす追加学習（Colab用）

Laya Multilingual の**エンコーダー本体**を日本語データで学習するノートブックです。これまでの laya-ja は本体を固定して採点層だけを学習していました。ここでは次の2方式を切り替えて試せます。

| `MODE` | 学習する部分 | 目安 |
|---|---|---|
| `"lora"`（初期値） | 全層の線形層に差し込む小さな差分行列＋判断ヘッド | 無料枠のT4で動く想定。まずこちら |
| `"full"` | 本体の全重み＋判断ヘッド | 効果は最大。メモリと時間が増える |

- 学習ループは Laya 本体の `laya.train`（上流の公式実装）をそのまま使います。
- データ分割は laya-ja の `experiment.partition` と同じ（乱数シード42、調整用192件、公式開発セット全件で評価）。`context_experiment.py` が取り置いた256問も学習から外し、別枠で評価します。
- 学習前後を同じ問題で比べ、結果を JSON に保存します。

**使い方**：メニューの「ランタイム → ランタイムのタイプを変更」で GPU（T4 など）を選び、上から順に実行してください。初回は動作確認として、設定セルの `MAX_TRAIN = 512` で一度通すことをおすすめします。

In [ ]:
# 1. GPU の確認
import torch

if not torch.cuda.is_available():
    raise SystemExit("GPU が見つかりません。「ランタイム → ランタイムのタイプを変更」で GPU を選んでください。")
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

In [ ]:
# 2. インストールとリポジトリの取得
#    PyTorch は Colab に入っている GPU 版をそのまま使う。
#    Laya は --no-deps で入れる（Vega 用の依存が別メジャーの Transformers を要求するため。install.py と同じ扱い）。
!pip install -q "transformers==4.57.6"
!pip install -q --no-deps "laya==0.4.31"

import os, sys

REPO_DIR = "/content/laya-ja"
REPO_COMMIT = "3db0cc4007775a5e4aa5445775a1f8a70b198c03"   # 分割と評価を固定するための版。新しい版で試すときは書き換える
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/nozomugit/laya-ja.git {REPO_DIR}
!git -C {REPO_DIR} checkout -q {REPO_COMMIT}
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
# 3. 保存先（Google ドライブ）
#    セッションが切れても結果が残るよう、ドライブに保存する。使わない場合は False。
SAVE_TO_DRIVE = True

OUT_ROOT = "/content/laya-ja-runs"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_ROOT = "/content/drive/MyDrive/laya-ja-runs"
    except Exception as e:
        print("ドライブを使わずに続行します:", e)
print("保存先:", OUT_ROOT)

In [ ]:
# 4. 設定
TASKS = ["jnli"]          # ["jnli", "jcommonsenseqa"] にすると2課題を混ぜて1つのモデルで学習する
MODE = "lora"             # "lora" = 全層LoRA ／ "full" = フル微調整
SEED = 42                 # 学習の乱数シード（データ分割は常に42で固定）
EPOCHS = 3
MICRO_BATCH = 16          # 1回の順伝播に載せる件数。メモリ不足なら 8 に
GRAD_ACCUM = 2            # 実効バッチ = MICRO_BATCH x GRAD_ACCUM
MAX_TRAIN = None          # 動作確認は 512 など。None で全件

LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

ENCODER_LR = {"lora": 2e-4, "full": 2.5e-5}[MODE]   # full の値は上流の既定値
HEAD_LR = 1e-4
LOSS = "soft-ce"          # "rlcd" にすると上流の既定の目的関数（方策勾配項つき）
SHUFFLE_OPTIONS = True    # 学習中、選択肢の並び順を毎回入れ替える（位置への依存を減らす）
GRAD_CHECKPOINT = False   # 長い文章で学習してメモリが足りないとき True
SAVE_FP32 = False         # True にすると完成版を fp32 で保存（約2倍の大きさ）

LOCAL_BASE = None         # 手元のチェックポイントを使う場合はそのディレクトリ。None で Hugging Face から取得

RUN_NAME = f"{'+'.join(TASKS)}-{MODE}-s{SEED}"
OUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
assert MODE in ("lora", "full") and set(TASKS) <= {"jnli", "jcommonsenseqa"} and TASKS
print(RUN_NAME, "->", OUT_DIR)

In [ ]:
# 5. データ（laya-ja と同じ分割）
import json
from pathlib import Path

import numpy as np

from experiment import bootstrap_difference, partition
from laya_ja import BASE, REVISION, nli_request
from scoring import metrics

DATA = Path(".cache/data")


def split(task):
    # context_experiment.samples と同じ手順。公式開発セットと画像ID・質問文が重なる例は partition が学習側から除く。
    d = partition(task, DATA, 42, 10**8, 192, 10**8)
    train, tune, dev = d["train"], d["tune"], d["eval"]
    used = {r[2] for r in train[:4096]}
    reserved = [r for r in train[4096:] if r[2] not in used][:256]
    reserved_groups = {r[2] for r in reserved}
    train = [r for r in train if r[2] not in reserved_groups]
    out = {"train": train[:MAX_TRAIN] if MAX_TRAIN else train, "tune": tune, "dev": dev, "reserved": reserved}
    if task == "jnli":
        probes = json.loads(Path("challenge.json").read_text())
        out["challenge"] = [(nli_request(p, h), y, cat, str(i)) for i, (cat, p, h, y) in enumerate(probes)]
    return out


ROWS = {task: split(task) for task in TASKS}
for task, s in ROWS.items():
    print(task, {name: len(rows) for name, rows in s.items()})

In [ ]:
# 6. ベースモデルの読み込み（版を固定し、重みファイルのハッシュを照合）
import hashlib

from laya.common import QTYPES, collate_items, uses_parallel_layout
from laya.train import (TrainConfig, encode_item, encode_state, load_checkpoint, make_item,
                        target_from_expected, to_internal, train_model)

BASE_SHA256 = "9d628fd971b700382ac6f65920a86f149777b2e748e0c955fb3b19695aa8f204"

if LOCAL_BASE:
    base_dir = LOCAL_BASE
else:
    from huggingface_hub import snapshot_download

    base_dir = snapshot_download(
        BASE, revision=REVISION,
        allow_patterns=["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"],
    )
    h = hashlib.sha256()
    with open(os.path.join(base_dir, "model.safetensors"), "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    assert h.hexdigest() == BASE_SHA256, "ベースモデルのハッシュが laya-ja の固定値と一致しません"

model, tok, cfg = load_checkpoint(base_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
max_len, head_max_len = cfg.get("max_len", 512), cfg.get("head_max_len", 192)
assert not uses_parallel_layout(cfg)
print(f"パラメータ数 {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M / max_len {max_len} / head_max_len {head_max_len}")

In [ ]:
# 7. 学習・評価用の入力を作る
def to_row(r):
    # laya-ja の (状態, 質問文, 選択肢) を、Laya 本体の学習形式 {state, questions, expected} に直す。
    (state, instruction, choices), gold, _, _ = r
    question = {"type": "choice", "instructions": instruction, "criteria": dict.fromkeys(choices, "")}
    return {"state": state, "questions": {"decision": question}, "expected": {"decision": choices[gold]}}


def make_items(rows, strict):
    # strict=True（評価用）は1件も落とさない。学習用は、選択肢がトークン上で区別できなくなる例などを除く。
    items, skipped = [], {}
    for r in rows:
        row = to_row(r)
        q = to_internal("decision", row["questions"]["decision"])
        target = target_from_expected(q, row["expected"]["decision"])
        state_ids = encode_state(tok, row["state"], max_len)
        item, reason = make_item(tok, q, target, state_ids, head_max_len, max_len)
        if item is None and strict:
            assert reason == "options_collapsed", f"評価用の例を入力に変換できません: {reason}"
            item = {"q": q, "state_ids": state_ids, "target": target, "qtype": QTYPES[q["t"]], "k": len(target)}
        if item is None:
            skipped[reason] = skipped.get(reason, 0) + 1
        else:
            items.append(item)
    if skipped:
        print("学習から除いた例:", skipped)
    return items


EVAL = {
    task: {name: (rows, make_items(rows, strict=True)) for name, rows in s.items() if name != "train"}
    for task, s in ROWS.items()
}
train_items = [it for task in TASKS for it in make_items(ROWS[task]["train"], strict=False)]
print("学習に使う例:", len(train_items))


@torch.no_grad()
def score(items, reverse=False, batch_size=64):
    # 各例について、選択肢ごとの採点（ロジット）を元の選択肢順で返す。reverse=True は選択肢を逆順に並べて入力する。
    was_training = model.training
    model.eval()
    order = sorted(range(len(items)), key=lambda i: len(items[i]["state_ids"]))
    out = [None] * len(items)
    for start in range(0, len(items), batch_size):
        idx = order[start:start + batch_size]
        chunk = [
            encode_item(tok, items[i], max_len, head_max_len,
                        list(range(items[i]["k"]))[::-1] if reverse else None)
            for i in idx
        ]
        b = collate_items([chunk], tok.pad_token_id)
        logits, _ = model(*(b[k].to(device) for k in
                            ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype")))
        for row, i in zip(logits.float().cpu(), idx):
            v = row[:items[i]["k"]]
            out[i] = v.flip(0) if reverse else v
    model.train(was_training)
    return torch.stack(out)


def labels(rows):
    return torch.tensor([r[1] for r in rows])


def score_all():
    result = {}
    for task, sets in EVAL.items():
        result[task] = {name: score(items) for name, (rows, items) in sets.items()}
        n = min(512, len(sets["dev"][1]))
        result[task]["dev-reversed"] = score(sets["dev"][1][:n], reverse=True)
    return result

In [ ]:
# 8. 学習前の成績（比較の基準）
base_scores = score_all()
for task in TASKS:
    for name, (rows, _) in EVAL[task].items():
        m = metrics(base_scores[task][name], labels(rows))
        print(f"{task:16s} {name:10s} 正答率 {m['accuracy']:.3f} ({m['correct']}/{m['n']})  NLL {m['nll']:.3f}")

In [ ]:
# 9. LoRA の差し込み（MODE = "lora" のときだけ）
import math

import torch.nn as nn
import torch.nn.functional as F


class LoRALinear(nn.Module):
    # 元の線形層 W は凍結し、W + (alpha/r)・B・A の B と A だけを学習する。B はゼロ初期化なので学習前の出力は元と同じ。
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.lora_A), self.lora_B) * self.scale

    def merged(self):
        with torch.no_grad():
            delta = (self.lora_B.float() @ self.lora_A.float()) * self.scale
            self.base.weight.add_(delta.to(self.base.weight.dtype))
        return self.base


def add_lora(encoder, r, alpha, dropout):
    for p in encoder.parameters():
        p.requires_grad_(False)
    targets = [n for n, m in encoder.named_modules() if isinstance(m, nn.Linear) and n.startswith("layers.")]
    for name in targets:
        parent_name, child = name.rsplit(".", 1)
        parent = encoder.get_submodule(parent_name)
        setattr(parent, child, LoRALinear(getattr(parent, child), r, alpha, dropout).to(device))
    return targets


def merge_lora(encoder):
    wrapped = [n for n, m in encoder.named_modules() if isinstance(m, LoRALinear)]
    for name in wrapped:
        parent_name, child = name.rsplit(".", 1)
        parent = encoder.get_submodule(parent_name)
        setattr(parent, child, getattr(parent, child).merged())
    for p in encoder.parameters():
        p.requires_grad_(True)


def apply_adapter(path):
    # 別のセッションで adapter.safetensors を使うとき：ベースモデルを読み込んだ直後（セル6の後）に呼ぶ。
    from safetensors import safe_open
    from safetensors.torch import load_file

    with safe_open(path, framework="pt") as f:
        meta = f.metadata()
    assert meta["base_revision"] == REVISION, "アダプターとベースモデルの版が一致しません"
    add_lora(model.encoder, int(meta["rank"]), float(meta["alpha"]), 0.0)
    result = model.load_state_dict(load_file(path), strict=False)
    assert not result.unexpected_keys, result.unexpected_keys
    model.eval()


if MODE == "lora":
    lora_targets = add_lora(model.encoder, LORA_RANK, LORA_ALPHA, LORA_DROPOUT)
    print("LoRA を差し込んだ線形層:", len(lora_targets))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"学習するパラメータ {trainable / 1e6:.1f}M / 全体 {total / 1e6:.0f}M ({100 * trainable / total:.1f}%)")

In [ ]:
# 10. 学習
#     各エポックの終わりに調整用192件で評価し、負の対数尤度が最小のエポックを採用する。
#     LoRA のときは、その時点のアダプターを保存先にも書き出す（セッション切れ対策）。
from safetensors.torch import save_file

config = TrainConfig(
    epochs=EPOCHS, micro_batch=MICRO_BATCH, grad_accum=GRAD_ACCUM,
    encoder_lr=ENCODER_LR, head_lr=HEAD_LR, loss=LOSS,
    shuffle_options=("choice",) if SHUFFLE_OPTIONS else (),
    gradient_checkpointing=GRAD_CHECKPOINT, seed=SEED, log_every=100,
)


def trainable_state():
    return {n: p.detach().float().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def save_adapter(state, path):
    meta = {"base": BASE, "base_revision": REVISION, "tasks": ",".join(TASKS), "mode": MODE,
            "rank": str(LORA_RANK), "alpha": str(LORA_ALPHA), "repo_commit": REPO_COMMIT}
    save_file({k: v.contiguous() for k, v in state.items()}, path, metadata=meta)


best = {"nll": float("inf"), "epoch": None, "state": None}
history = []


def on_epoch_end(epoch, loss):
    tune = {}
    for task in TASKS:
        rows, items = EVAL[task]["tune"]
        tune[task] = metrics(score(items), labels(rows))
    nll = sum(m["nll"] for m in tune.values()) / len(tune)
    history.append({"epoch": epoch + 1, "train_loss": loss, "tune_nll": nll,
                    "tune_accuracy": {t: m["accuracy"] for t, m in tune.items()}})
    print(f"  調整用: NLL {nll:.4f} / 正答率 " + ", ".join(f"{t} {m['accuracy']:.3f}" for t, m in tune.items()), flush=True)
    if nll < best["nll"]:
        best.update(nll=nll, epoch=epoch + 1, state=trainable_state())
        if MODE == "lora":
            save_adapter(best["state"], os.path.join(OUT_DIR, "adapter.safetensors"))


train_losses = train_model(model, tok, train_items, config, device, max_len, head_max_len, on_epoch_end=on_epoch_end)
model.load_state_dict(best["state"], strict=False)
model.eval()
print("採用したエポック:", best["epoch"])

In [ ]:
# 11. 学習後の評価と、学習前との比較
tuned_scores = score_all()
report = {
    "run": RUN_NAME, "mode": MODE, "tasks": TASKS, "seed": SEED, "base": BASE, "base_revision": REVISION,
    "repo_commit": REPO_COMMIT, "train_items": len(train_items), "selected_epoch": best["epoch"],
    "history": history,
    "config": {"epochs": EPOCHS, "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM, "encoder_lr": ENCODER_LR,
               "head_lr": HEAD_LR, "loss": LOSS, "shuffle_options": SHUFFLE_OPTIONS,
               "lora": {"rank": LORA_RANK, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT} if MODE == "lora" else None},
    "results": {},
}
for task in TASKS:
    res = report["results"][task] = {}
    for name, (rows, _) in EVAL[task].items():
        y = labels(rows)
        before, after = base_scores[task][name], tuned_scores[task][name]
        diff = (after.argmax(-1) == y).int() - (before.argmax(-1) == y).int()
        res[name] = {
            "base": metrics(before, y), "tuned": metrics(after, y),
            "gained": int((diff == 1).sum()), "lost": int((diff == -1).sum()),
            # 同じ問題の学習前後を対応づけ、画像ID・質問文などのグループ単位で再標本化した95%区間
            "accuracy_difference_95pct": bootstrap_difference(diff.numpy(), np.array([r[2] for r in rows])),
        }
    rows, _ = EVAL[task]["dev"]
    y = labels(rows)
    n = len(tuned_scores[task]["dev-reversed"])
    res["order_reversed_agreement"] = {
        k: float((s[task]["dev"][:n].argmax(-1) == s[task]["dev-reversed"].argmax(-1)).float().mean())
        for k, s in (("base", base_scores), ("tuned", tuned_scores))
    }
    if task == "jnli":
        res["label_recall"] = {
            label: {k: float((s[task]["dev"][y == i].argmax(-1) == i).float().mean())
                    for k, s in (("base", base_scores), ("tuned", tuned_scores))}
            for i, label in enumerate(("entailment", "contradiction", "neutral"))
        }

Path(OUT_DIR, "results.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")

for task in TASKS:
    print(f"\n== {task} ==")
    for name in EVAL[task]:
        r = report["results"][task][name]
        lo, hi = r["accuracy_difference_95pct"]
        print(f"{name:10s} {r['base']['accuracy']:.3f} -> {r['tuned']['accuracy']:.3f} "
              f"({r['tuned']['correct']}/{r['tuned']['n']})  差の95%区間 {100 * lo:+.1f}〜{100 * hi:+.1f}pt  "
              f"ECE {r['base']['ece_10_bins']:.3f} -> {r['tuned']['ece_10_bins']:.3f}")
    r = report["results"][task]
    print("選択肢を逆順にしたときの一致率:", {k: round(v, 3) for k, v in r["order_reversed_agreement"].items()})
    if "label_recall" in r:
        print("ラベル別再現率:", {k: {a: round(b, 3) for a, b in v.items()} for k, v in r["label_recall"].items()})

In [ ]:
# 12. 保存
#     checkpoint/ … LoRA を本体に織り込んだ完成版。laya.load(パス) でそのまま読める
#     adapter.safetensors … LoRA の差分と判断ヘッドだけの小さなファイル（MODE = "lora" のとき）
#     保存したものを読み直して開発セットを再評価し、保存前と同じ成績になるかを確かめる。
if MODE == "lora":
    merge_lora(model.encoder)
    merged = score(EVAL[TASKS[0]]["dev"][1])
    print("織り込み前後の採点の最大差:", float((merged - tuned_scores[TASKS[0]]["dev"]).abs().max()))

CKPT_DIR = os.path.join(OUT_DIR, "checkpoint")
os.makedirs(CKPT_DIR, exist_ok=True)
dtype = torch.float32 if SAVE_FP32 else torch.float16     # 上流の配布形式は fp16
save_file({k: v.detach().to(dtype).cpu().contiguous() for k, v in model.state_dict().items()},
          os.path.join(CKPT_DIR, "model.safetensors"))
model.encoder.config.save_pretrained(os.path.join(CKPT_DIR, "encoder"))
tok.save_pretrained(os.path.join(CKPT_DIR, "tokenizer"))
Path(CKPT_DIR, "rl_agent_config.json").write_text(
    json.dumps(dict(cfg, max_len=max_len, head_max_len=head_max_len), indent=2))

del model
torch.cuda.empty_cache()
model, _, _ = load_checkpoint(CKPT_DIR)
model.to(device)
report["saved_checkpoint_check"] = {}
for task in TASKS:
    rows, items = EVAL[task]["dev"]
    reloaded = metrics(score(items), labels(rows))
    before_save = report["results"][task]["dev"]["tuned"]
    report["saved_checkpoint_check"][task] = {"in_memory": before_save["accuracy"], "reloaded": reloaded["accuracy"]}
    print(f"{task}: 保存前 {before_save['accuracy']:.4f} / 読み直し後 {reloaded['accuracy']:.4f}")
    if abs(reloaded["accuracy"] - before_save["accuracy"]) > 0.005:
        print("  差が0.5ポイントを超えています。SAVE_FP32 = True にして保存し直してください。")
Path(OUT_DIR, "results.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print("保存先:", OUT_DIR, sorted(os.listdir(OUT_DIR)))

In [ ]:
# 13. 保存したモデルを Laya の通常の入口から使ってみる
import laya
from laya_ja import query

agent = laya.load(CKPT_DIR, device=str(device))
if "jnli" in TASKS:
    # 正解は「判断できない」（README の例。初版の公開アダプターは「矛盾」と誤判定していた）
    print(query(agent, *nli_request("整理券の所持者は全員入場できる。秋さんは持っていない。", "秋さんは入場できる。")))
    print(query(agent, *nli_request("二人が横断歩道を渡っている。", "人が道を渡っている。")))
if "jcommonsenseqa" in TASKS:
    from laya_ja import qa_request
    print(query(agent, *qa_request("鉛筆で書いた線を消すために使うものは？", ["定規", "消しゴム", "封筒", "電卓", "切手"])))

## 結果の読み方と次の手順

- **比較対象**：セル11の `dev` が公式開発セット全件です。README の表（JNLI は上流52.2%、初版72.1%、文脈モデル84.8%）と同じ問題で比べられます。ただしこちらは GPU 上の評価なので、CPU で評価した既存の値とは小数点以下がずれることがあります。リポジトリに載せる最終値は、保存した `checkpoint/` を CPU で評価し直してください。
- **`reserved`** は学習にもモデル選定にも使っていない256問、**`challenge`** は独自36問（JNLI のみ）です。開発セットを繰り返し見て設定を選ぶと値が楽観的になるので、設定の比較は `tune` と `reserved` で行い、`dev` は最後の確認に使うのが安全です。
- **手元で使う**：`laya.load("checkpoint のパス")` で読み込み、`laya_ja.query` に渡します。`laya_ja.load("jnli")` と `predict.py jnli` は、初版の採点層の重みを上書きで読み込み、ベースモデルのハッシュも照合するため、この完成版には使えません。
- **アダプターだけを使う**：`adapter.safetensors` は、別のセッションでセル6まで実行したあと `apply_adapter(パス)` を呼ぶと適用できます（セル9の定義が必要です）。
- **確率の校正はやり直していません**。保存した設定の温度はベースモデルのままです。確信度を閾値に使う場合は、調整用データで温度を合わせ直してください。
- **次に試すこと**：`TASKS = ["jnli", "jcommonsenseqa"]` での混合学習、`SEED` を変えた3回の実行、`MODE = "full"` との比較。JSTS（score 型）と JCoLA（noul 型）は `to_row` に型ごとの変換を足せば同じ学習ループに載せられます。